# 🇨🇲 Cameroon SmartTrip — Simulation Colab

Cette partie teste le comportement de l'agent sur de nombreux profils.

**Ce qui est simulé :** budget, nombre de voyageurs, durée et préférences.  
**Ce qui reste réel/traçable :** hôtels, transports, activités et tarifs du projet.


## 1. Importer le ZIP complet

In [ ]:
from google.colab import files
import zipfile, shutil, sys
from pathlib import Path

print("Importez Projet_03_Cameroon_SmartTrip_Colab_Complet_AVEC_SIMULATION.zip")
uploaded = files.upload()
zip_name = next(name for name in uploaded if name.lower().endswith(".zip"))

root = Path("/content/smarttrip")
if root.exists():
    shutil.rmtree(root)
root.mkdir()

with zipfile.ZipFile(zip_name, "r") as z:
    z.extractall(root)

matches = list(root.rglob("Cameroon-SmartTrip-Colab"))
PROJECT_DIR = matches[0] if matches else next(p for p in root.iterdir() if p.is_dir())
APP_DIR = PROJECT_DIR / "app"
sys.path.insert(0, str(APP_DIR))

print("Projet :", PROJECT_DIR)


## 2. Installer les dépendances

In [ ]:
import subprocess, sys
subprocess.check_call([
    sys.executable, "-m", "pip", "install", "-q",
    "-r", str(PROJECT_DIR / "requirements_colab.txt")
])
print("Dépendances installées.")


## 3. Simuler 30 profils

In [ ]:
from simulation import run_profile_simulation, summarize_simulation
import pandas as pd

simulations = run_profile_simulation(seed=42, n=30)
display(simulations.head(10))


## 4. KPI de simulation

In [ ]:
summary = summarize_simulation(simulations)

print("Nombre de simulations :", summary["n_simulations"])
print("Taux de plans dans le budget :", f"{summary['success_rate']:.1%}")
print("Coût moyen :", f"{summary['average_total_fcfa']:,.0f} FCFA")

display(summary["top_hotels"])
display(summary["top_transport"])


## 5. Résultats par tranche de budget

In [ ]:
simulations["budget_band"] = pd.cut(
    simulations["budget_fcfa"],
    bins=[0,150000,250000,350000,10_000_000],
    labels=["≤150k","150–250k","250–350k",">350k"],
    include_lowest=True
)

budget_summary = (
    simulations.groupby("budget_band", observed=False)
    .agg(
        simulations=("simulation_id","count"),
        success_rate=("within_budget","mean"),
        avg_total=("total_fcfa","mean")
    )
    .reset_index()
)
display(budget_summary)


## 6. Graphique de faisabilité

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.bar(
    budget_summary["budget_band"].astype(str),
    budget_summary["success_rate"] * 100
)
plt.xlabel("Tranche de budget")
plt.ylabel("Plans dans le budget (%)")
plt.title("Faisabilité des séjours selon le budget")
plt.ylim(0,100)
plt.show()


## 7. Hôtels les plus retenus

In [ ]:
hotel_counts = simulations["hotel"].value_counts().head(10)

plt.figure(figsize=(10,5))
plt.barh(hotel_counts.index[::-1], hotel_counts.values[::-1])
plt.xlabel("Nombre de sélections")
plt.title("Hôtels les plus souvent sélectionnés par l'agent")
plt.show()


## 8. Stress test budgétaire

Même profil, mêmes préférences, mais budget variable.
C'est cette partie qui montre comment la décision de l'agent évolue.


In [ ]:
from simulation import run_budget_stress_test

stress = run_budget_stress_test(
    origin="Douala",
    travelers=4,
    nights=2,
    budgets=[100000,125000,150000,175000,200000,225000,250000,300000,350000,450000,600000],
    preferences=("Plage","Piscine","Wi-Fi","Famille"),
    include_lobe=True
)

display(stress[[
    "budget_fcfa","status","hotel","transport_class",
    "total_fcfa","remaining_fcfa","adjustments"
]])


## 9. Courbe du stress test

In [ ]:
plt.figure(figsize=(10,5))
plt.plot(stress["budget_fcfa"], stress["total_fcfa"], marker="o", label="Coût du plan")
plt.plot(stress["budget_fcfa"], stress["budget_fcfa"], linestyle="--", label="Budget")
plt.xlabel("Budget disponible (FCFA)")
plt.ylabel("FCFA")
plt.title("Stress test budgétaire")
plt.legend()
plt.show()


## 10. Changements de décision

In [ ]:
changes = stress[[
    "budget_fcfa","hotel","transport_class","adjustments"
]].copy()

changes["hotel_change"] = changes["hotel"].ne(changes["hotel"].shift())
changes["transport_change"] = changes["transport_class"].ne(changes["transport_class"].shift())

display(changes)


## 11. Votre propre simulation

In [ ]:
my_stress = run_budget_stress_test(
    origin="Yaoundé",
    travelers=2,
    nights=3,
    budgets=[150000,200000,250000,300000,400000],
    preferences=("Plage","Restaurant","Petit budget"),
    include_lobe=False
)

display(my_stress[[
    "budget_fcfa","status","hotel","transport_class",
    "total_fcfa","remaining_fcfa"
]])


## 12. Télécharger les résultats

In [ ]:
from google.colab import files
from pathlib import Path

p1 = Path("/content/simulation_profils.csv")
p2 = Path("/content/stress_test_budget.csv")

simulations.to_csv(p1, index=False, encoding="utf-8-sig")
stress.to_csv(p2, index=False, encoding="utf-8-sig")

files.download(str(p1))
files.download(str(p2))


# Conclusion

La simulation permet d'évaluer le comportement du prototype et constitue la **partie expérimentale** du projet.
